# 第4章 同步电机模型 (Synchronous Machine Models)

> 参考教材：Kundur P. *Power System Stability and Control*, McGraw-Hill, 1994, Chapter 4

## 本章概述

本章介绍三种不同精度的同步电机模型，对应于不同的时间尺度和分析需求：

| 模型 | 状态变量 | 精度 | 适用场景 |
|------|---------|------|---------|
| **经典模型** (Classical) | δ, ω (2阶) | 低 | 暂态稳定初步分析 |
| **暂态模型** (Transient) | δ, ω, Eq' (3阶) | 中 | 计及磁链衰减的稳定分析 |
| **次暂态模型** (Subtransient) | δ, ω, Eq', Ed', Eq'', Ed'' (6阶) | 高 | 短路电流精确计算 |

本章例题覆盖 Kundur 教材中的 Example 4.1-4.3。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from psa4teaching.models import Generator, GeneratorModelType

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
print('导入成功！')

In [ ]:
# ===== 同步电机参数 (Kundur Example 4.1-4.3) =====
# 555 MVA, 24 kV, 60 Hz
gen = Generator(
    bus=1, name='G1',
    Xd=1.81, Xd_prime=0.30, Xd_doubleprime=0.22,
    Xq=1.76, Xq_prime=0.65, Xq_doubleprime=0.25,
    Td0_prime=8.0, Td0_doubleprime=0.03,
    Tq0_prime=0.4, Tq0_doubleprime=0.05,
    H=6.5, D=0.0,
    Sb=555.0, Vb=24.0
)

# 运行条件: P=0.9, Q=0.3 (过励), Vt=1.0∠0°
P, Q, Vt = 0.9, 0.3, 1.0
S = P + 1j*Q
Ia = np.conj(S) / Vt

print(f'电机参数: {gen.Sb:.0f} MVA, {gen.Vb:.0f} kV')
print(f'Xd={gen.Xd}, Xd\'={gen.Xd_prime}, Xd"={gen.Xd_doubleprime}')
print(f'Xq={gen.Xq}, Xq\'={gen.Xq_prime}, Xq"={gen.Xq_doubleprime}')
print(f'\n运行条件: P={P:.1f} pu, Q={Q:.1f} pu, Vt={Vt:.1f}∠0° pu')
print(f'电枢电流: |Ia| = {abs(Ia):.4f} pu, φ = {np.degrees(np.angle(Vt)-np.angle(Ia)):.2f}°')

---
## 4.1 稳态模型 — Example 4.1

### 电压源在同步电抗 Xd 之后

稳态模型中，发电机表示为一个理想电压源 $\\bar{E}$ 串联同步电抗 $X_d$：

$$\\bar{E} = \\bar{V}_t + jX_d \\bar{I}_a$$

**适用条件：**
- 稳态潮流分析
- 不考虑磁链衰减
- 假设励磁电流恒定

**局限性：**
- 无法反映暂态过程中的磁场变化
- 在短路等大扰动下误差很大

In [ ]:
# ===== Example 4.1: 稳态模型 =====
print('='*50)
print('Example 4.1: 稳态模型 (电压源在 Xd 之后)')
print('='*50)

# 虚构电势 (q轴方向)
Eq_ss = Vt + 1j * gen.Xq * Ia
delta_ss = np.angle(Eq_ss)
print(f'\n虚构电势: Eq = {abs(Eq_ss):.4f}∠{np.degrees(delta_ss):.2f}° pu')

# dq轴分解
Id_ss = abs(Ia) * np.sin(delta_ss - np.angle(Ia))
Iq_ss = abs(Ia) * np.cos(delta_ss - np.angle(Ia))

# 稳态励磁电势 (电压源在 Xd 之后)
Vq_ss = Vt * np.cos(delta_ss)
Efd_ss = Vq_ss + gen.Xd * Id_ss

print(f'功角: δ = {np.degrees(delta_ss):.2f}°')
print(f'dq 轴电流: Id = {Id_ss:.4f} pu, Iq = {Iq_ss:.4f} pu')
print(f'励磁电势 (稳态): Efd = {Efd_ss:.4f} pu')
print(f'\n稳态模型等值电路: Efd ∠ δ —[jXd]— Vt ∠ 0°')

---
## 4.2 暂态模型 — Example 4.2

### 电压源在暂态电抗 Xd' 之后

暂态模型中，发电机表示为暂态电势 $\\bar{E}'$ 串联暂态电抗 $X_d'$：

$$\\bar{E}' = \\bar{V}_t + jX_d' \\bar{I}_a$$

**特点：**
- 计及励磁绕组磁链守恒（$\\psi_{fd}$ 不能突变）
- 暂态时间尺度 (0.5-5s) 内精度良好
- 是暂态稳定分析中最常用的模型

**与稳态模型的区别：**
- $E'$ < $E_{fd}$（因为 Xd' < Xd）
- 适用于故障后 ~0.1-5s 的时间窗口

In [ ]:
# ===== Example 4.2: 暂态模型 =====
print('='*50)
print('Example 4.2: 暂态模型 (电压源在 Xd\' 之后)')
print('='*50)

# 方法1: 使用 Generator 内置方法
E_prime_builtin = gen.compute_transient_emf(Vt, Ia)
print(f'\n【方法1: 内置方法】')
print(f'E\' = Vt + jXd\'·Ia = {E_prime_builtin.real:.4f} + j{E_prime_builtin.imag:.4f}')
print(f'|E\'| = {abs(E_prime_builtin):.4f} pu')

# 方法2: 手动计算（用于验证）
E_prime_manual = Vt + 1j * gen.Xd_prime * Ia
print(f'\n【方法2: 手动计算】')
print(f'E\' = {E_prime_manual.real:.4f} + j{E_prime_manual.imag:.4f}')
print(f'|E\'| = {abs(E_prime_manual):.4f} pu')

# 对比稳态和暂态
print(f'\n【对比分析】')
print(f'稳态电势 (在 Xd 之后):   |Efd| = {abs(Efd_ss):.4f} pu')
print(f'暂态电势 (在 Xd\' 之后):  |E\'|  = {abs(E_prime_manual):.4f} pu')
print(f'差值: ΔE = {abs(Efd_ss) - abs(E_prime_manual):.4f} pu')
print(f'原因: Xd ({gen.Xd:.2f}) >> Xd\' ({gen.Xd_prime:.2f})，因此在 Xd\'后的压降更小')

---
## 4.3 次暂态模型 — Example 4.3

### 电压源在次暂态电抗 Xd'' 之后

次暂态模型中，发电机表示为次暂态电势 $\\bar{E}''$ 串联次暂态电抗 $X_d''$：

$$\\bar{E}'' = \\bar{V}_t + jX_d'' \\bar{I}_a$$

**特点：**
- 计及阻尼绕组效应
- 适用于短路后 ~0.01-0.1s 的初始阶段
- 是短路电流计算的标准模型

**三种模型对照：**
$$
\\underbrace{E''}_{\\text{次暂态}} < \\underbrace{E'}_{\\text{暂态}} < \\underbrace{E_{fd}}_{\\text{稳态}}
$$
$$
\\underbrace{X_d''}_{\\text{次暂态}} < \\underbrace{X_d'}_{\\text{暂态}} < \\underbrace{X_d}_{\\text{稳态}}
$$

In [ ]:
# ===== Example 4.3: 次暂态模型 =====
print('='*50)
print('Example 4.3: 次暂态模型 (电压源在 Xd" 之后)')
print('='*50)

# 使用 Generator 内置方法
E_doubleprime = gen.compute_subtransient_emf(Vt, Ia)
print(f'\nE" = Vt + jXd"·Ia = {E_doubleprime.real:.4f} + j{E_doubleprime.imag:.4f}')
print(f'|E"| = {abs(E_doubleprime):.4f} pu')

# 手动验证
E_doubleprime_manual = Vt + 1j * gen.Xd_doubleprime * Ia
print(f'\n手动验证: |E"| = {abs(E_doubleprime_manual):.4f} pu')

# 三种电势对比
print(f'\n【三种模型对比】')
print(f'{"模型":<15} {"电抗":>8} {"电势幅值":>10}')
print('-'*40)
print(f'{"稳态 (Steady)":<15} {gen.Xd:>8.2f} {abs(Efd_ss):>10.4f} pu')
print(f'{"暂态 (Transient)":<15} {gen.Xd_prime:>8.2f} {abs(E_prime_manual):>10.4f} pu')
print(f'{"次暂态 (Subtrans)":<15} {gen.Xd_doubleprime:>8.2f} {abs(E_doubleprime):>10.4f} pu')
print(f'\n结论: X" < X\' < Xd, E" < E\' < Efd（电抗越小，背后电势越小）')

In [ ]:
# ===== 三种模型相量图对比 =====
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

models = [
    ('稳态模型 (Xd={:.2f})'.format(gen.Xd), gen.Xd, abs(Efd_ss), 'blue'),
    ('暂态模型 (Xd\'={:.2f})'.format(gen.Xd_prime), gen.Xd_prime, abs(E_prime_manual), 'red'),
    ('次暂态模型 (Xd"={:.2f})'.format(gen.Xd_doubleprime), gen.Xd_doubleprime, abs(E_doubleprime), 'green'),
]

for ax, (title, X, E_mag, color) in zip(axes, models):
    # 计算电势
    E = Vt + 1j * X * Ia
    delta_i = np.angle(E)
    
    # 绘制 Vt
    ax.arrow(0, 0, 0, Vt, head_width=0.04, head_length=0.05,
             fc='black', ec='black', linewidth=2)
    ax.text(-0.15, Vt/2, '$V_t$', fontsize=11, ha='right')
    
    # 绘制 E
    Ex, Ey = E.real, E.imag
    ax.arrow(0, 0, Ex, Ey, head_width=0.04, head_length=0.05,
             fc=color, ec=color, linewidth=2)
    ax.text(Ex*1.1, Ey*1.1, f'|E|={E_mag:.3f}', fontsize=10, color=color)
    
    # 绘制 jX·Ia
    if Ex > 0:
        ax.arrow(0, Vt, Ex, Ey-Vt, head_width=0.03, head_length=0.03,
                 fc='gray', ec='gray', linewidth=1.5, linestyle='--')
    
    # 功角弧
    arc_r = 0.5
    theta_arc = np.linspace(0, delta_i, 30)
    ax.plot(arc_r*np.sin(theta_arc), arc_r*np.cos(theta_arc), color, linewidth=1)
    ax.text(arc_r*1.2*np.sin(delta_i/2), arc_r*1.2*np.cos(delta_i/2),
            f'$\\delta$={np.degrees(delta_i):.1f}°', fontsize=9)
    
    ax.set_xlim(-0.5, 2.5)
    ax.set_ylim(-0.2, 2.0)
    ax.set_xlabel('实轴 (pu)')
    ax.set_ylabel('虚轴 (pu)')
    ax.set_title(title, fontsize=13)
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')

plt.suptitle('三种同步电机模型的相量图对比 (P=0.9, Q=0.3, Vt=1.0)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('examples/kundur/three_models_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4.4 模型选择指南

| 分析类型 | 推荐模型 | 时间窗口 |
|---------|---------|---------|
| 潮流计算 | 稳态 (Xd) | 稳态 |
| 暂态稳定 | 暂态 (Xd') | 0.1-5 s |
| 短路电流 | 次暂态 (Xd'') | 0-0.1 s |
| 小干扰稳定 | 暂态 (Xd') + 励磁系统 | 频域分析 |
| 电磁暂态 | 次暂态 (Xd'', Xq'') + 阻尼绕组 | 0-0.02 s |

**关键结论：**
1. 电抗越小 → 适用范围越接近故障瞬间
2. 电势越小 → 反映了磁场不能突变的物理本质
3. 对于暂态稳定分析，$X_d'$ 模型是最佳平衡点（精度 vs 计算量）

---
## 本章小结

1. **稳态模型 (Xd)**：最简单的模型，适用于潮流计算和稳态分析
2. **暂态模型 (Xd')**：计及励磁绕组磁链守恒，是暂态稳定分析的标准模型
3. **次暂态模型 (Xd'')**：计及阻尼绕组效应，适用于短路电流的精确计算
4. **模型选择**：取决于分析的时间尺度和精度要求

> 下一章：[第12章 小干扰稳定分析](./ch12a_smib_small_signal.ipynb) — 基于 Heffron-Phillips 模型的 SMIB 特征值分析